# NB2 — Train MIL head on cached embeddings

**Settings:** GPU T4 x2 / P100, Internet OFF.
**Attach:** dataset `rsna-knee-embeddings-v1` (NB1 output), dataset `rsna-knee-weak-labels` (train_weak_v2.csv), competition data (for train.csv only).
Output: `best.pt` (~25 MB) → publish as dataset `rsna-knee-ckpt-v1`.

In [ ]:
import os, glob
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score


IGNORE = {'StudyInstanceUID', 'Report'}
WPS = 42          # windows per study in embeddings
HIDDEN = 128
EPOCHS = 12
LR = 1e-3
BATCH = 16
FOLD_FRAC = 0.2   # unused with team folds (kept for reference)
VAL_FOLD = 0      # Daksh canonical folds: hold out fold 0
SEED = 42
comp_cands = glob.glob('/kaggle/input/*/train.csv') + glob.glob('/kaggle/input/*/*/train.csv') + glob.glob('/kaggle/input/*/*/*/train.csv')
assert comp_cands, f'train.csv not found; mounts={sorted(os.listdir("/kaggle/input"))}'
COMP = Path(os.path.dirname(comp_cands[0]))
emb_cands = (glob.glob('/kaggle/input/*/embeddings') + glob.glob('/kaggle/input/*/*/embeddings') + glob.glob('/kaggle/input/*/*/*/embeddings'))
emb_cands = [c for c in emb_cands if glob.glob(c + '/*.npz')]
assert emb_cands, f'embeddings not found; mounts={sorted(os.listdir("/kaggle/input"))}'
EMB = Path(emb_cands[0])
lab_cands = glob.glob('/kaggle/input/*/train_weak_v2.csv') + glob.glob('/kaggle/input/*/*/train_weak_v2.csv') + glob.glob('/kaggle/input/*/*/*/train_weak_v2.csv')
assert lab_cands, f'train_weak_v2.csv not found; mounts={sorted(os.listdir("/kaggle/input"))}'
LABELS = Path(lab_cands[0])
print('COMP =', COMP); print('EMB =', EMB); print('LABELS =', LABELS)
fold_cands = glob.glob('/kaggle/input/*/folds.csv') + glob.glob('/kaggle/input/*/*/folds.csv') + glob.glob('/kaggle/input/*/*/*/folds.csv')
assert fold_cands, f'folds.csv not found; mounts={sorted(os.listdir("/kaggle/input"))}'
FOLDS = Path(fold_cands[0])
tgt_cands = glob.glob('/kaggle/input/*/v1_targets.csv') + glob.glob('/kaggle/input/*/*/v1_targets.csv') + glob.glob('/kaggle/input/*/*/*/v1_targets.csv')
assert tgt_cands, f'v1_targets.csv not found; mounts={sorted(os.listdir("/kaggle/input"))}'
TARGETS = Path(tgt_cands[0])
print('FOLDS =', FOLDS); print('TARGETS =', TARGETS)


In [ ]:
class EmbDataset(Dataset):
    def __init__(self, rows, label_cols, emb_dir):
        self.rows = rows.reset_index(drop=True)
        self.label_cols = label_cols
        self.emb_dir = Path(emb_dir)
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows.iloc[i]
        uid = str(r['StudyInstanceUID'])
        f = self.emb_dir / f'{uid}.npz'
        emb = np.load(f)['emb'].astype(np.float32)
        w = emb.shape[0]
        if w >= WPS:
            emb = emb[:WPS]
        else:
            emb = np.concatenate([emb] * int(np.ceil(WPS / w)), axis=0)[:WPS]
        y = r[self.label_cols].to_numpy(dtype=np.float32)
        mask = ~np.isnan(y)
        y = np.nan_to_num(y).astype(np.float32)
        return torch.from_numpy(emb), torch.from_numpy(y), torch.from_numpy(mask), uid

In [ ]:
class GatedAttentionMIL(nn.Module):
    def __init__(self, in_features=384, hidden_dim=128, num_classes=12):
        super().__init__()
        self.attention_v = nn.Sequential(nn.Linear(in_features, hidden_dim), nn.Tanh())
        self.attention_u = nn.Sequential(nn.Linear(in_features, hidden_dim), nn.Sigmoid())
        self.attention_weights = nn.Linear(hidden_dim, 1)
        self.classifier = nn.Linear(in_features, num_classes)
    def forward(self, h):
        v = self.attention_v(h); u = self.attention_u(h)
        a = self.attention_weights(v * u)
        w = torch.softmax(a, dim=1)
        m = torch.sum(w * h, dim=1)
        return self.classifier(m)

In [ ]:
df_gold = pd.read_csv(COMP / 'train.csv')
label_cols = [c for c in df_gold.columns if c not in IGNORE]

folds = pd.read_csv(FOLDS)
df = pd.read_csv(TARGETS)
assert list(df.columns[1:]) == label_cols, (df.columns.tolist(), label_cols)
df = df.merge(folds[['StudyInstanceUID', 'fold']], on='StudyInstanceUID', how='left')
assert df['fold'].notna().all(), 'fold missing for some studies'

val_idx = df.index[df['fold'] == VAL_FOLD].to_numpy()
tr_idx = df.index[df['fold'] != VAL_FOLD].to_numpy()
gold_in_val = int(((folds['fold'] == VAL_FOLD) & (folds['is_gold'] == 1)).sum())
print(f'train {len(tr_idx)}  val {len(val_idx)}  (gold in val fold: {gold_in_val})')

In [ ]:
device = 'cuda'
bce = nn.BCEWithLogitsLoss(reduction='none')

def loss_fn(logits, y, mask):
    l = bce(logits, y)
    return (l * mask).sum() / mask.sum().clamp(min=1)

def train_fold(fold):
    folds_arr = df['fold'].to_numpy()
    va_idx = df.index[folds_arr == fold].to_numpy()
    tr_idx = df.index[folds_arr != fold].to_numpy()
    torch.manual_seed(SEED + fold)
    tr_loader = DataLoader(EmbDataset(df.loc[tr_idx], label_cols, EMB), batch_size=BATCH, shuffle=True, num_workers=2, generator=torch.Generator().manual_seed(SEED + fold))
    va_loader = DataLoader(EmbDataset(df.loc[va_idx], label_cols, EMB), batch_size=BATCH, shuffle=False, num_workers=2)
    model = GatedAttentionMIL(384, HIDDEN, len(label_cols)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    best_auc, best_sd = 0.0, None
    for ep in range(1, EPOCHS + 1):
        model.train(); tl = 0
        for emb, y, mask, _ in tr_loader:
            emb, y, mask = emb.to(device), y.to(device), mask.to(device)
            opt.zero_grad()
            loss = loss_fn(model(emb), y, mask)
            loss.backward(); opt.step()
            tl += loss.item() * len(emb)
        model.eval(); ys, ps, ms = [], [], []
        with torch.no_grad():
            for emb, y, mask, _ in va_loader:
                p = torch.sigmoid(model(emb.to(device))).cpu().numpy()
                ys.append(y.numpy()); ps.append(p); ms.append(mask.numpy())
        Y, P, M = np.concatenate(ys), np.concatenate(ps), np.concatenate(ms)
        yb = (Y > 0.5).astype(int)
        aucs = []
        for j in range(Y.shape[1]):
            ok = M[:, j] > 0
            if ok.sum() > 4 and len(np.unique(yb[ok, j])) > 1:
                aucs.append(roc_auc_score(yb[ok, j], P[ok, j]))
        macro = float(np.mean(aucs))
        print(f'fold {fold} epoch {ep}: loss {tl/len(tr_idx):.4f} val-macro-AUC {macro:.4f}', flush=True)
        if macro > best_auc:
            best_auc = macro
            best_sd = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    return va_idx, best_auc, best_sd

oof = np.full((len(df), len(label_cols)), np.nan, dtype=np.float32)
state_dicts, oof_aucs = [], []
for fold in range(5):
    va_idx, best_auc, best_sd = train_fold(fold)
    state_dicts.append(best_sd)
    oof_aucs.append(best_auc)
    model = GatedAttentionMIL(384, HIDDEN, len(label_cols)).to(device)
    model.load_state_dict(best_sd); model.eval()
    va_loader = DataLoader(EmbDataset(df.loc[va_idx], label_cols, EMB), batch_size=BATCH, shuffle=False, num_workers=2)
    ps = []
    with torch.no_grad():
        for emb, y, mask, _ in va_loader:
            ps.append(torch.sigmoid(model(emb.to(device))).cpu().numpy())
    oof[va_idx] = np.concatenate(ps)
    print(f'fold {fold} best val-macro-AUC {best_auc:.4f}', flush=True)

oof_val = float(np.mean(oof_aucs))
torch.save({'state_dicts': state_dicts, 'label_cols': label_cols, 'hidden': HIDDEN, 'val_auc': oof_val}, '/kaggle/working/best.pt')
pd.DataFrame(oof, columns=label_cols).assign(StudyInstanceUID=df['StudyInstanceUID'].to_numpy()).to_csv('/kaggle/working/oof_preds.csv', index=False)
print('ENSEMBLE OOF val-macro-AUC', round(oof_val, 4))

In [ ]:
import json
# ---- gold-58 scoring (ensemble mean of 5 fold models) ----
from sklearn.metrics import roc_auc_score

best = torch.load('/kaggle/working/best.pt', map_location='cpu', weights_only=False)
models = []
for sd in best['state_dicts']:
    m = GatedAttentionMIL(384, HIDDEN, len(label_cols)).to(device)
    m.load_state_dict(sd); m.eval()
    models.append(m)

gold_uids = df_gold['StudyInstanceUID'].astype(str).tolist()
preds = []
with torch.no_grad():
    for uid in gold_uids:
        f = EMB / f'{uid}.npz'
        emb = np.load(f)['emb'].astype(np.float32)
        w = emb.shape[0]
        if w >= WPS:
            emb = emb[:WPS]
        else:
            emb = np.concatenate([emb] * int(np.ceil(WPS / w)), axis=0)[:WPS]
        x = torch.from_numpy(emb).unsqueeze(0).to(device)
        p = np.mean([torch.sigmoid(m(x)).cpu().numpy()[0] for m in models], axis=0)
        preds.append(p)

P = np.stack(preds)
gdf = df_gold.set_index('StudyInstanceUID').loc[gold_uids]
Y = gdf[label_cols].to_numpy(dtype=np.float32)
aucs = []
for j, col in enumerate(label_cols):
    ok = ~np.isnan(Y[:, j])
    if ok.sum() > 4 and len(np.unique(Y[ok, j])) > 1:
        aucs.append((col, roc_auc_score(Y[ok, j], P[ok, j])))
for c, a in aucs:
    print(f'  {c:20s} {a:.4f}')
gold_macro = float(np.mean([a for _, a in aucs]))
print('GOLD-58 MACRO AUC (5-fold ensemble):', round(gold_macro, 4))

# honest out-of-fold gold score: each gold study predicted by the model that never saw it
oof_df = pd.read_csv('/kaggle/working/oof_preds.csv').set_index('StudyInstanceUID')
O = oof_df.loc[gold_uids, label_cols].to_numpy(dtype=np.float32)
oaucs = []
for j, col in enumerate(label_cols):
    ok = ~np.isnan(Y[:, j])
    if ok.sum() > 4 and len(np.unique(Y[ok, j])) > 1:
        oaucs.append(roc_auc_score(Y[ok, j], O[ok, j]))
oof_gold = float(np.mean(oaucs))
print('GOLD-58 MACRO AUC (strict OOF):', round(oof_gold, 4))

pd.DataFrame(P, index=gold_uids, columns=label_cols).rename_axis('StudyInstanceUID').to_csv('/kaggle/working/gold_preds.csv')
json.dump({'gold_macro_auc': gold_macro, 'oof_gold_macro_auc': oof_gold, 'soft_val_auc': best['val_auc']}, open('/kaggle/working/gold_score.json', 'w'), indent=1)
print('wrote gold_preds.csv + oof_preds.csv + gold_score.json')